# Neural Language Models: From Bengio's NPLM to Recurrent Networks

Topic 3's models estimate a next-word distribution separately for every context they have counted. This
notebook replaces counts with **coordinates**. Each word gets a learned vector, a context becomes a point,
and a softmax turns the point into a distribution, so that nearby contexts predict alike. That is Bengio's
NPLM. Training it is maximum likelihood, the gradient is computed by reverse accumulation, and a recurrent
state lifts the trigram's two-word horizon, up to the point where gradients through time vanish.

Every number is computed by `neural_language_models.py`, which owns them. The models are small and written
from scratch in NumPy: an NPLM (d = 64, h = 128) and an Elman RNN (64/128), trained by Adam on Topic 3's
69,312-token training text. Following Topic 4's promise, every comparison is reported in **bits per
character** of an honest open-vocabulary code: an out-of-vocabulary word is `<unk>` followed by its spelling,
under the same speller for every model.

In [ ]:
import math
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "neural_language_models.py").exists()
                       else here / "notebooks" / "neural-language-models"))
import neural_language_models as nl

pce, ng = nl.pce, nl.ng
for t in ("train", "dev", "held", "edgar"):
    s = nl.corpus()[t]
    print(f"{t:5}: {len(s):5d} sentences, {pce.n_word_tokens(s):6d} predicted tokens, {pce.n_chars(s):7d} characters")
print(f"|V| = {nl.V}; dev = generate_corpus({nl.DEV_DOCS}, {nl.DEV_SEED}) chooses epochs, mixture weights, and the "
      f"dynamic-evaluation step; held-out text tunes nothing")

## 1. From counts to coordinates

**Definition 1 (NPLM; Bengio et al. 2003).** An embedding table $E \in \mathbb{R}^{(|V|+1)\times d}$ (one extra
row for `<s>`) maps the context $u = (u_1, u_2)$ to the point $x(u) = [E_{u_1}; E_{u_2}] \in \mathbb{R}^{2d}$. Then
$a = \tanh(xW_1 + b_1)$, $z = aW_2 + b_2$, and $p(\cdot \mid u) = \mathrm{softmax}(z)$.

**Proposition 1.** Every word gets positive probability in every context, and each $p(\cdot\mid u)$ sums to 1.
Topic 3's zero-count problem cannot occur, by construction.

**Theorem 1 (smoothing by geometry).** For all context points $x, x'$ and every word $w$,
$$|\log p(w\mid x) - \log p(w\mid x')| \le \sqrt2\,\|W_2\|_2\,\|W_1\|_2\,\|x - x'\|_2 .$$

*Proof.* $\nabla_x \log p(w\mid x) = W_1\,\mathrm{diag}(1-a^2)\,W_2\,(e_w - p)$ in column form. Here
$\|e_w - p\|_2^2 = (1-p_w)^2 + \sum_{j\ne w}p_j^2 \le 2(1-p_w)^2 \le 2$ and $0 < 1 - a_k^2 \le 1$. Apply the mean value
theorem on the segment, then Cauchy–Schwarz. $\blacksquare$

Kneser–Ney's contexts are atoms: two contexts share information only through a common suffix. The NPLM's
contexts are points, and its predictions move continuously with them.

In [ ]:
nplm = nl.trained_nplm()
print("NPLM: dev-selected epoch", nplm.epoch + 1, "| per-epoch nats/token:",
      [{k: round(v, 4) for k, v in row.items()} for row in nplm.history])
nl.test_softmax_is_positive_and_proper()
nl.test_lipschitz_smoothing_random()
nl.test_lipschitz_smoothing_corpus()
r = nl.lipschitz_ratios()
print(f"Thm 1 constant L = {nl.nplm_lipschitz(nplm.params):.1f}; over {len(r):,} distinct held-out context pairs the observed "
      f"ratio max_w|Δ log p| / (L‖Δx‖) has median {np.median(r):.4f} and max {r.max():.4f}")

Where does geometry beat counting? We split the held-out tokens by what training saw. Class 2 tokens have
their trigram in training, class 1 tokens only their bigram, and class 0 tokens neither.

In [ ]:
nl.test_sharing_wins_where_counts_are_absent()
b = nl.class_breakdown()
for c, label in ((0, "bigram unseen"), (1, "bigram seen, trigram unseen"), (2, "trigram seen")):
    print(f"class {c} ({label:27}, {b['count'][c]:5d} tokens): KN {b['kn'][c]:.3f}  NPLM {b['nplm'][c]:.3f}  "
          f"RNN {b['rnn'][c]:.3f} nats")

## 2. Training is maximum likelihood

The NPLM is trained by minimizing held-in cross-entropy. By Topic 4's Corollary 1 that is maximum likelihood,
under a strictly proper score.

**Proposition 2 (the softmax is the categorical exponential family).** With $A(z) = \log\sum_j e^{z_j}$,
$\nabla A = p$ and $\nabla^2 A = \partial p/\partial z = \mathrm{diag}(p) - pp^\top = \mathrm{Cov}_p(e_Y)$ (formalStatistics,
*Exponential Families*, Thm 7.1). It is PSD, because $v^\top(\mathrm{diag}\,p - pp^\top)v = \mathrm{Var}_p(v_Y)$, with null space
$\mathrm{span}(\mathbb 1)$, so its rank is $K-1$. Hence $\ell(z) = -\log p_y = A(z) - z_y$ is convex, with gradient $p - e_y$.

**Theorem 2 (score identity and moment matching).** For fixed features $a_i$ and $z_i = a_iW + b$, the mean loss
$L$ has $\nabla_W L = \tfrac1n\sum_i a_i^\top(p_i - e_{y_i})$ and $\nabla_b L = \tfrac1n\sum_i(p_i - e_{y_i})$, and $L$ is convex in $(W, b)$.
At any stationary point, $\tfrac1n\sum_i p_i = \tfrac1n\sum_i e_{y_i}$: the model's average prediction equals the
empirical distribution of the targets. This extends formalStatistics' canonical-link score identity (*GLMs*,
Thm 1) to $K$ classes. It holds for the whole network too, since $b_2$ is one of its parameters.

In [ ]:
nl.test_softmax_exponential_family()
nl.test_score_identity_and_moment_matching()
print("Prop 2 and Thm 2 hold on random instances (complex-step Jacobians; L-BFGS stationary points)")

**Corollary 1 (a never-seen target has no MLE).** If word $w$ never occurs as a target, then
$\partial L/\partial b_w = \tfrac1n\sum_i p_i(w) > 0$ at every parameter value (Prop 1). So no stationary point
exists, and gradient descent pushes $p(w \mid \cdot)$ toward 0. Topic 3's zero-count problem returns one level up.
The softmax is positive, but fitting it by maximum likelihood is not smoothing.

`<unk>` is such a word: the raw training text has none. The cure is Topic 3's Good–Turing logic. Words seen once
stand in for words never seen: in every epoch, each token of a once-seen word becomes `<unk>` with
probability ½ (word dropout). Theorem 2 then predicts that the trained model's mean $p(\texttt{<unk>})$ matches
the target rate.

In [ ]:
nl.test_zero_frequency_target_has_no_mle()
u = nl.unk_moment()
print(f"with word dropout: mean p(<unk>) over training contexts {u['mean_p_unk']:.4f}; target rate in one draw "
      f"{u['target_rate']:.4f}; expected rate ½·n1/N = {u['expected_rate']:.4f}")
for name, label in (("nplm-nodrop", "no dropout, 1 epoch"), ("nplm", "word dropout")):
    for t in nl.TESTS:
        c = nl.unk_cost(name, t)
        print(f"  {label:20} {t:5}: mean p(<unk>) {c['mean_p_unk']:.2e}, cost per <unk> {c['unk_nats']:.2f} nats, "
              f"{c['nats']:.4f} nats/token")
print(f"1/|V| = {1 / nl.V:.2e}; uniform log-loss {pce.uniform_log_loss():.4f} nats")

**Proposition 3 (the network is not convex).** Relabeling hidden units leaves the function, and so the loss,
unchanged. If $L$ were convex, the midpoint of $\theta$ and its relabeling $\pi\theta$ would have loss at most
$L(\theta)$. The trained NPLM violates this, so formalML's convex convergence rates for gradient descent do not
apply: training finds a stationary point at best, and nothing certifies that it is a global optimum.

In [ ]:
nl.test_network_is_not_convex()
c = nl.nonconvexity()
print(f"first 4,096 training events (one word-dropout draw): L(θ) = {c['loss']:.4f}, L(πθ) = {c['loss_permuted']:.4f}, L((θ+πθ)/2) = {c['loss_midpoint']:.4f}")

## 3. Backpropagation is reverse accumulation

**Definition 2.** A computational graph lists nodes $v_1,\dots,v_N$ in topological order. Each non-input node is
$v_i = \varphi_i(v_{\mathrm{pa}(i)})$, and $v_N$ is a scalar. The **adjoint** $\bar v_i$ is the derivative of $v_N$ with respect to
$v_i$, with the earlier nodes held fixed and the later nodes recomputed.

**Theorem 3 (reverse accumulation).**
- (i) $\bar v_N = 1$ and $\bar v_i = \sum_{c\,:\,i\in\mathrm{pa}(c)} \bar v_c\, \partial\varphi_c/\partial v_i$, so one sweep in reverse topological
order gives every adjoint.
- (ii) If each primitive's vector–Jacobian products cost at most $\omega$ times the primitive, the sweep costs
at most $\omega$ times the forward pass, plus one addition per edge and coordinate. For $Y = XW$, the products
$\bar YW^\top$ and $X^\top\bar Y$ each cost what $XW$ does, so $\omega = 2$.

*Proof of (i).* Let $F_i(v_1,\dots,v_i)$ run the program from node $i+1$ on. Then
$F_{i-1} = F_i(\,\cdot\,, \varphi_i(\cdot))$. By the multivariate chain rule (formalCalculus, *The Jacobian*, Thm 2),
$\partial F_{i-1}/\partial v_j = \partial F_i/\partial v_j + (\partial F_i/\partial v_i)(\partial\varphi_i/\partial v_j)$, which is the update the sweep makes when it
processes node $i$. By induction, once nodes $N,\dots,i+1$ are processed, the accumulator of every remaining node $v_j$, $j \le i$, holds
$\partial F_i/\partial v_j$; in particular the accumulator for $v_i$ holds $\partial F_i/\partial v_i = \bar v_i$. $\blacksquare$

We check (i) on 200 random graphs (198 with fan-out), and every hand-written network gradient, against the **complex
step** $\partial_j f \approx \operatorname{Im} f(\theta + ih\,e_j)/h$. That derivative has no subtractive cancellation, so with
$h = 10^{-30}$ it is exact to rounding.

In [ ]:
nl.test_reverse_accumulation_random_dags()
nl.test_nplm_gradient()
print("Thm 3 on 200 random DAGs and Cor 2 (the NPLM's hand-written gradient) agree with complex-step derivatives")

## 4. Recurrent networks and gradients through time

**Definition 3 (Elman RNN language model).** $h_0 = 0$ and $h_t = \tanh(x_tW_x + h_{t-1}W_h + b)$, where $x_t$ embeds the
previous word (`<s>` first). The next word's distribution is $\mathrm{softmax}(h_tW_o + b_o)$. The state summarizes the whole
prefix, so no Markov horizon is imposed.

**Theorem 4 (tightness).** Let $\varepsilon = \big(\sum_j \exp(\|W_o[:,j] - W_o[:,\text{eos}]\|_1 + b_{o,j} - b_{o,\text{eos}})\big)^{-1}$. Then
$p(\texttt{</s>}\mid h) \ge \varepsilon$ on the whole cube $\|h\|_\infty \le 1$. Hence $P(N > n) \le (1-\varepsilon)^n$ for the length $N$ of a sampled
sentence, the probabilities of finite sentences sum to 1, and every moment of $N$ is finite.

*Proof.* By Hölder, $z_j - z_{\text{eos}} \le \|h\|_\infty\|W_o[:,j]-W_o[:,\text{eos}]\|_1 + b_{o,j} - b_{o,\text{eos}}$, and
$p_{\text{eos}} = 1/\sum_j e^{z_j - z_{\text{eos}}}$. Next, $P(N > n) = \mathbb E\prod_{t\le n}(1 - p(\texttt{</s>}\mid h_t)) \le (1-\varepsilon)^n$. $\blacksquare$

**Corollary 3 (BPTT).** Theorem 3 on the unrolled graph. Tied weights receive the sum of their per-step gradients,
and the gradient reaching $h_k$ from step $t$ passes through $\partial h_t/\partial h_k$.

In [ ]:
rnn = nl.trained_rnn()
print("RNN: dev-selected epoch", rnn.epoch + 1, "| per-epoch nats/token:",
      [{k: round(v, 4) for k, v in row.items()} for row in rnn.history])
nl.test_rnn_gradient()
nl.test_rnn_tight()
R = rnn.params
print(f"Thm 4 for the trained RNN: ε = {nl.tightness_epsilon(R['Wo'], R['bo'], nl.EOS_ID):.2e} "
      "(positive, so the model is tight; as a numerical tail bound it is vacuous)")

**Theorem 5 (Jacobian products).** $\partial h_t/\partial h_k = \prod_{j=t}^{k+1}\mathrm{diag}(1-h_j^2)\,W_h^\top$, so
$$\Big\|\frac{\partial h_t}{\partial h_k}\Big\|_2 \le \prod_{j=k+1}^{t}\big\|\mathrm{diag}(1-h_j^2)W_h^\top\big\|_2 \le \sigma_{\max}(W_h)^{\,t-k}.$$
- (a) If $\sigma_{\max} < 1$, the gradient vanishes geometrically.
- (b) Explosion requires $\sigma_{\max} > 1$.
- (c) For the linear recurrence, $\|W_h^m\|_2 \ge \rho(W_h)^m$ and $\|W_h^m\|_2^{1/m} \to \rho(W_h)$ (Gelfand), so the gradient
explodes once $\rho > 1$.

*Proof.* Chain rule; submultiplicativity; $\|\mathrm{diag}(1-h^2)\|_2 \le 1$. For (c), an eigenpair gives $\|W^mv\| = |\lambda|^m\|v\|$. $\blacksquare$

The trained network meets neither (a) nor the linear picture, and every per-step factor has norm above 1, yet
its gradients vanish. Saturated units make $1 - h_j^2$ small, and successive factors' expanding directions do not
line up. The decay lives in the products, where no per-step norm can see it.

In [ ]:
nl.test_jacobian_bound_random()
nl.test_jacobian_bound_corpus()
print(f"trained RNN: σ_max(W_h) = {np.linalg.norm(R['Wh'], 2):.3f}, ρ(W_h) = {np.abs(np.linalg.eigvals(R['Wh'])).max():.3f}")
cd = nl.corpus_decay()
print(f"per-step factors ‖diag(1 − h²)W_hᵀ‖: min {cd['min_step_norm']:.3f}, median {cd['median_step_norm']:.3f}; "
      f"{100 * cd['saturated_share']:.1f}% of units have |h| > 0.9")
for m, x, bd, n in zip(cd["lags"], cd["measured"], cd["bound"], cd["n"]):
    print(f"  lag {m:2d}: median ‖∂h_T/∂h_(T-m)‖ = {x:7.4f}   trajectory bound {bd:10.1f}   ({n} sentences)")

**Definition 4 (LSTM).** Gates $i, f, o$ (sigmoid) and $g$ (tanh) read $(x_t, h_{t-1})$, with
$c_t = f_t\odot c_{t-1} + i_t\odot g_t$ and $h_t = o_t\odot\tanh c_t$.

**Proposition 4 (the cell path).** $\partial c_t/\partial c_k = \prod_{j=k+1}^{t}\mathrm{diag}(f_j) + R_{t,k}$. Every term of $R$ contains a
factor $\partial c_j/\partial h_{j-1}$, which is linear in the recurrent weights of gates $i, f, g$. In particular, with those weights at 0,
the derivative is exactly $\prod_j\mathrm{diag}(f_j)$, a product with no weight matrix in it. With $f \equiv 1$ this is
Hochreiter and Schmidhuber's constant error carousel.

*Proof.* Expand the product of the $(c, h)$ block Jacobians over paths. The gates do not read $c$, so the only
block leading from $h$ back to $c$ is $\partial c_j/\partial h_{j-1}$. $\blacksquare$

So the forget bias sets how far gradients reach at initialization: about $\sigma(b_f)^{t-k}$. We test this on a bit
that must be remembered across 60 noise symbols.

In [ ]:
nl.test_lstm_cell_path()
nl.test_lag_gradient()
nl.test_long_lag_memory()
for name, xs in nl.long_lag_census().items():
    print(f"{name:7}: median gradient reaching the bit at initialization {np.median(xs):.2e} (12 initializations)")
for name, row in nl.long_lag_training().items():
    print(f"{name:7}: after {nl.LAG_STEPS} Adam steps (one start), test accuracy {row['accuracy']:.3f}, loss {row['test_loss']:.4f}")
print(f"cell-path prediction at lag {nl.LAG}: σ(1)^60 = {(1 / (1 + math.exp(-1))) ** 60:.1e}, "
      f"σ(3)^60 = {(1 / (1 + math.exp(-3))) ** 60:.3f}")

## 5. The softmax bottleneck

**Theorem 6 (Yang et al. 2018).** Suppose the logits are $z(u) = s(u)W + b$ with a state $s(u) \in \mathbb{R}^d$. On any $N$
contexts, the log-probability matrix is $L = SW + \mathbb 1 b^\top - \lambda\mathbb 1^\top$, where $\lambda_i$ is the log-sum-exp of row
$i$. So $\operatorname{rank} L \le d + 2$, and its row-centered version has rank at most $d + 1$. A target matrix
none of whose row shifts has rank $\le d+1$ cannot be represented exactly. A mixture of softmaxes escapes the bound.

In [ ]:
nl.test_softmax_bottleneck()
s = nl.bottleneck_summary()
print(f"over {s['contexts']} contexts: rank NPLM {s['rank_nplm']} (≤ h + 2 = {s['h'] + 2}), RNN {s['rank_rnn']}, "
      f"interpolated KN {s['rank_kn']}; KN's row-centered matrix has {100 * s['kn_energy_beyond']:.2f}% of its "
      f"squared Frobenius norm beyond rank {s['h'] + 1}")

## 6. The honest comparison

Each model's held-out score is a ratio estimate over sentences (Topic 4, Thm 2), reported per token and in
bits per character of the open-vocabulary code. All models share Topic 3's vocabulary and Topic 4's speller,
so the spelling bits are identical across rows. The mixtures $\lambda p_{\text{neural}} + (1-\lambda)p_{\text{KN}}$ take $\lambda$ from dev.
**Dynamic evaluation** takes one SGD step on each sentence right after scoring it.

In [ ]:
nl.test_neural_beats_kn_in_domain()
print(f"λ: NPLM {nl.mix_lambda('nplm')}, RNN {nl.mix_lambda('rnn')}; dynamic-evaluation step {nl.dynamic_lr()}")
print(f"parameters: NPLM {nl.param_count(nplm.params):,}, RNN {nl.param_count(R):,}; KN stores {nl.kn_table_size():,} n-gram counts")
for t in nl.TESTS:
    print(f"\n{t}: model       nats/token (se)    PP       bpc (se)          bpc − KN (paired se)")
    for name in nl.COMPARED:
        r = nl.comparison_row(name, t)
        print(f"   {name:9} {r['nats']:7.4f} ({r['nats_se']:.4f}) {r['pp']:8.2f}  {r['bpc']:.4f} ({r['bpc_se']:.4f})  "
              f"{r['bpc_minus_kn']:+.4f} ({r['bpc_minus_kn_se']:.4f})")

**Proposition 5 (dynamic evaluation is a code).** Topic 4's arithmetic coder needs only a sequential model
$Q(x_i \mid x_{<i})$. Parameters that are a deterministic function of the text already coded qualify, because the
decoder recomputes them from what it has decoded. This is prequential coding. The coder below takes
`dynamic_update`'s step after every sentence, and the decoder replays it.

In [ ]:
nl.test_dynamic_evaluation_is_a_code()
for t in nl.TESTS:
    c = nl.corpus_code("rnn-dyn", t)
    n = pce.n_chars(nl.corpus()[t])
    print(f"{t:5}: one codeword of {c.length:,} bits (−log2 Q = {c.model_bits:,.2f}) for {c.n_symbols:,} symbols "
          f"= {c.length / n:.4f} bpc")
print(f"Topic 4's interpolated KN code of the held-out text: {pce.corpus_code('ikn', 'held').length:,} bits")

## 7. Finance case study: EDGAR

Out of domain, 57.6% of EDGAR's tokens are `<unk>`. The `<unk>` events carry about half or more of each static model's loss,
so in-domain training now hurts. At the dev-selected epoch, EDGAR is worse than after the first epoch, while
dev is better. Dynamic evaluation learns EDGAR's own `<unk>` rate as it codes. One floor remains.

**Proposition 6 (the spelling floor).** In any code that sends an OOV word as `<unk>` plus its spelling under $S$,
the bits per character are at least $\sum_{\mathrm{OOV}} -\log_2 S(w) / n_{\text{char}}$.

In [ ]:
nl.test_finance_findings()
f = nl.finance_summary()
print(f"uniform: {f['uniform_nats']:.4f} nats/token; lzma given the training text: {f['lzma_edgar_bpc']:.4f} bpc")
for name in nl.COMPARED:
    print(f"  {name:9} {f[name + '_edgar_nats']:.4f} nats/token  {f[name + '_edgar_bpc']:.4f} bpc")
for m in ("nplm", "rnn"):
    h = (nl.trained_nplm() if m == "nplm" else nl.trained_rnn()).history
    print(f"{m}: EDGAR by epoch {[round(r['edgar'], 3) for r in h]}, dev by epoch {[round(r['dev'], 3) for r in h]}")
print(f"spelling alone: {f['spelled_bpc_edgar']:.4f} bpc (> lzma); with a PPM-C cache {f['cache_spelled_bpc_edgar']:.4f}; "
      f"dynamic RNN + cache {f['rnn_dyn_cache_bpc_edgar']:.4f} bpc — lzma still wins")

## The lab's numbers, and the full harness

`viz_payload()` holds every number the lab shows. Panel A has three sentences under KN, NPLM, RNN, and the
dynamic RNN, with per-token bits, event classes, OOV spellings, and the loss's sensitivity to every earlier
state. Panel B has synthetic gradient-through-time curves over a grid of $\sigma_{\max}(W_h)$, the trained RNN's
measured decay, and the long-lag results.

In [ ]:
nl.test_viz_payload()
P = nl.viz_payload()
for s in P["sentences"]:
    tot = {m: round(sum(t["bits"][m] for t in s["tokens"]), 1) for m in P["models"]}
    print(" ".join(s["words"]), f"({s['test']}):", tot, "bits before spelling")
print("σ grid:", P["through_time"]["sigmas"], "median ρ:", P["through_time"]["rho"])

In [ ]:
assert nl._run_harness() == 0